# Addition von Schallquellen

Vergleich von kohärenter und inkohärenter Addition
zweier Schallquellen an verschiedenen Empfängerpositionen.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoeverc/laermarmesysteme_medien/blob/main/Python/VL2/AdditionSchallquellen.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ===========================================================
# Einstellungen
# ===========================================================
fontsize=14
linewidth=2

f=np.arange(1, 1001, dtype=float)  # Frequenzen, Hz
Lp_sources=70
p0=2e-5
peff_sources=10**(Lp_sources/20)*p0

c=340
k=2*np.pi*f/c

# Quell- und Empfaengerkoordinaten
source_coords=np.array([[1, 8], [1, 2]],
  dtype=float)
rec_coords=np.array([[8, 8], [3, 7], [8, 3]],
  dtype=float)

# ===========================================================
# Berechnung: Schalldruckpegel an den Empfaengern
# ===========================================================
# Abstaende Quelle -> Empfaenger
r=np.zeros((2, 3))
for ii in range(2):
  for jj in range(3):
    dx=source_coords[ii, 0]-rec_coords[jj, 0]
    dy=source_coords[ii, 1]-rec_coords[jj, 1]
    r[ii, jj]=np.sqrt(dx**2+dy**2)

# Kohaerente Addition (Druckamplituden addieren)
p_coh=np.zeros((3, len(f)))
for jj in range(3):
  p_coh[jj, :]=(
    peff_sources
    *np.cos(2*np.pi*f-k*r[0, jj])/r[0, jj]
    +peff_sources
    *np.cos(2*np.pi*f-k*r[1, jj])/r[1, jj]
  )
Lp_coh=10*np.log10(p_coh**2/p0**2)

# Inkohaerente Addition (Leistungen addieren)
p_inc_sq=np.zeros((3, len(f)))
for jj in range(3):
  p_inc_sq[jj, :]=np.ones(len(f))*(
    peff_sources**2/r[0, jj]**2
    +peff_sources**2/r[1, jj]**2
  )
Lp_inc=10*np.log10(p_inc_sq/p0**2)

# ===========================================================
# Plot
# ===========================================================
fig=plt.figure(figsize=(14, 10))
gs=GridSpec(3, 5, figure=fig)

# Geometrie
ax_map=fig.add_subplot(gs[:, :3])
ax_map.plot(
  source_coords[:, 0], source_coords[:, 1],
  '*k', markersize=10)
ax_map.plot(
  rec_coords[:, 0], rec_coords[:, 1], 'sr')
for ii in range(2):
  ax_map.text(
    source_coords[ii, 0],
    source_coords[ii, 1]-0.3,
    f'Quelle {ii+1}', fontsize=fontsize)
for ii in range(3):
  ax_map.text(
    rec_coords[ii, 0],
    rec_coords[ii, 1]-0.3,
    f'Empfaenger {ii+1}', fontsize=fontsize)
ax_map.set_xlim(0, 10)
ax_map.set_ylim(0, 10)
ax_map.set_xlabel('x in m', fontsize=fontsize)
ax_map.set_ylabel('y in m', fontsize=fontsize)
ax_map.set_aspect('equal')
ax_map.legend(
  ['Quellen', 'Empfaenger'], fontsize=fontsize)

# Frequenzspektren an den 3 Empfaengern
for jj in range(3):
  ax=fig.add_subplot(gs[jj, 3:])
  ax.semilogx(
    f, Lp_coh[jj, :], linewidth=linewidth,
    label='Kohaerent')
  ax.semilogx(
    f, Lp_inc[jj, :], linewidth=linewidth,
    label='Inkohaerent')
  ax.set_ylim(
    np.min(Lp_coh[jj, :])-1,
    np.max(Lp_coh[jj, :])+6)
  ax.set_xlim(f[0], f[-1])
  ax.set_xlabel('f in Hz')
  ax.set_ylabel(r'$L_p$ in dB re 2e-5 Pa')
  ax.legend(loc='lower left')
  ax.set_title(
    f'Empfaenger {jj+1}', fontsize=fontsize)

plt.tight_layout()
plt.show()